In [ ]:
import subprocess
import os

LOCAL_DATA_DIR = "data_deteccion"
os.makedirs(f"{LOCAL_DATA_DIR}/images", exist_ok=True)
os.makedirs(f"{LOCAL_DATA_DIR}/labels", exist_ok=True)

BUCKET = "s3://humanoid-damage-cnn-834257582282-us-east-1-an/processed/damage_classification"

subprocess.run(["aws", "s3", "sync", f"{BUCKET}/images/", f"{LOCAL_DATA_DIR}/images/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/labels/", f"{LOCAL_DATA_DIR}/labels/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/classes.txt", f"{LOCAL_DATA_DIR}/classes.txt", "--quiet"], check=True)
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/data.yaml", f"{LOCAL_DATA_DIR}/data.yaml", "--quiet"], check=True)

n_images = len(os.listdir(f"{LOCAL_DATA_DIR}/images"))
n_labels = len(os.listdir(f"{LOCAL_DATA_DIR}/labels"))
print(f"Imágenes: {n_images} | Labels: {n_labels}")


In [ ]:
from collections import Counter
import glob

CLASSES_PATH = f"{LOCAL_DATA_DIR}/classes.txt"
with open(CLASSES_PATH) as f:
    CLASS_NAMES_FULL = [line.strip() for line in f if line.strip()]

#Nos quedamos con estas clases
CLASS_NAMES = [
    "Diagonal crack",
    "Concrete spalling",
    "Corrosion",
    "Shear crack",
    "Flexural/rocking crack",
    "Debris",
    "Detachment crack",
]
N_CLASSES = len(CLASS_NAMES)

# class_id original -> class_id nuevo (0..N_CLASSES-1). Si no está, es clase excluida.
CLASS_ID_MAP = {
    CLASS_NAMES_FULL.index(nombre): nuevo_id
    for nuevo_id, nombre in enumerate(CLASS_NAMES)
}

print(f"Clases en classes.txt: {len(CLASS_NAMES_FULL)}")
print(f"Clases que vamos a usar: {N_CLASSES}")
for nuevo_id, nombre in enumerate(CLASS_NAMES):
    print(f"  {nuevo_id}: {nombre}")

# Conteo de ejemplos por clase, ya filtrado.
conteo = Counter()
for label_path in glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"):
    with open(label_path) as f:
        for line in f:
            partes = line.split()
            if len(partes) != 5:
                continue
            class_id = CLASS_ID_MAP.get(int(partes[0]))
            if class_id is None:
                continue  # clase excluida
            conteo[CLASS_NAMES[class_id]] += 1

print("\nConteo de daños por clase:")
for clase, cantidad in conteo.most_common():
    print(f"  {clase}: {cantidad}")
print(f"\nTotal de daños anotados usados: {sum(conteo.values())}")


In [ ]:
import numpy as np

# Pesos de rebalanceo por clase: inversamente proporcional a la frecuencia,
# normalizado para que la clase más común tenga peso 1. Las clases raras
# pesan más en la loss (detection_loss, más abajo) y se sobremuestrean más
# en el split de train (repeticiones_por_rareza, más abajo).
max_conteo = max(conteo[c] for c in CLASS_NAMES)
CLASS_WEIGHTS = np.array([max_conteo / conteo[c] for c in CLASS_NAMES], dtype=np.float32)

print("Pesos por clase:")
for nombre, peso in zip(CLASS_NAMES, CLASS_WEIGHTS):
    print(f"  {nombre}: {peso:.2f}")


In [ ]:
# Ya no usamos una grilla: cada imagen predice un número FIJO de "slots" de
# caja (MAX_BOXES). Necesitamos que MAX_BOXES sea >= al máximo de daños reales
# (de las 7 clases elegidas) que puede tener una sola imagen, si no perdemos anotaciones.
max_danos_en_imagen = 0
for label_path in glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"):
    with open(label_path) as f:
        n = 0
        for line in f:
            partes = line.split()
            if len(partes) != 5:
                continue
            if CLASS_ID_MAP.get(int(partes[0])) is not None:
                n += 1
    max_danos_en_imagen = max(max_danos_en_imagen, n)

MAX_BOXES = max_danos_en_imagen + 5  # margen extra sobre el máximo observado
print(f"Máximo de daños anotados (de las 7 clases) en una sola imagen: {max_danos_en_imagen}")
print(f"MAX_BOXES elegido: {MAX_BOXES}")


In [ ]:
def parse_label_to_boxes(label_path, max_boxes=MAX_BOXES, n_classes=N_CLASSES):
    """Convierte un archivo de labels (formato YOLO) en un target de tamaño
    FIJO (max_boxes, 1+4+n_classes): cada fila es un daño real (objectness=1,
    con su caja y clase) o padding (objectness=0, resto en cero). A diferencia
    de la grilla, acá xc/yc/w/h quedan en coordenadas absolutas de la imagen
    completa (0-1), no relativas a ninguna celda: no hay celdas."""
    target = np.zeros((max_boxes, 1 + 4 + n_classes), dtype=np.float32)
    if not os.path.exists(label_path):
        return target  # imagen sin ningún daño anotado

    entries = []
    with open(label_path) as f:
        for line in f:
            partes = line.split()
            if len(partes) != 5:
                continue
            class_id = CLASS_ID_MAP.get(int(partes[0]))
            if class_id is None:
                continue  # clase excluida
            xc, yc, w, h = map(float, partes[1:5])
            entries.append((xc, yc, w, h, class_id))

    entries = entries[:max_boxes]  # si hay más daños que max_boxes, se descartan los sobrantes
    for i, (xc, yc, w, h, class_id) in enumerate(entries):
        target[i, 0] = 1.0
        target[i, 1:5] = [xc, yc, w, h]
        target[i, 5 + int(class_id)] = 1.0
    return target


### Cómo se ven las cajas reales

Verificamos que `parse_label_to_boxes` decodifica bien las anotaciones,
dibujando el cuadrado rojo + nombre de clase directo desde el `.txt`.

In [ ]:
#PARA PROBAR
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def plot_real_box_on_image(label_path, ax):
    """Dibuja las cajas reales anotadas, directo en coordenadas absolutas de
    la imagen (sin celdas de por medio)."""
    boxes = parse_label_to_boxes(label_path)
    valid = boxes[:, 0] == 1

    img_name = os.path.basename(label_path).replace(".txt", "")
    img_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = f"{LOCAL_DATA_DIR}/images/{img_name}{ext}"
        if os.path.exists(candidate):
            img_path = candidate
            break

    img = Image.open(img_path)
    img_w, img_h = img.size
    ax.imshow(img)

    for xc, yc, w, h, *class_onehot in boxes[valid, 1:]:
        box_w = w * img_w
        box_h = h * img_h
        x0 = xc * img_w - box_w / 2
        y0 = yc * img_h - box_h / 2

        rect = patches.Rectangle((x0, y0), box_w, box_h,
                                  linewidth=2, edgecolor="red",
                                  facecolor="none")
        ax.add_patch(rect)
        class_idx = int(np.argmax(class_onehot))
        ax.text(x0, y0 - 4, CLASS_NAMES[class_idx],
                color="red", fontsize=7, weight="bold")

    ax.set_title(img_name, fontsize=9)
    ax.axis("off")


# Elegimos 2 labels random que tengan al menos un daño anotado
labels_con_dano = []
for label_path in glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"):
    boxes = parse_label_to_boxes(label_path)
    if boxes[:, 0].sum() > 0:
        labels_con_dano.append(label_path)

muestras = random.sample(labels_con_dano, 2)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, label_path in zip(axes, muestras):
    plot_real_box_on_image(label_path, ax)

plt.tight_layout()
plt.show()


In [ ]:
image_files = sorted(glob.glob(f"{LOCAL_DATA_DIR}/images/*.jpg"))

def get_label_path(image_path):
    return image_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"

MAX_REPEATS = 6  # tope de repeticiones por imagen: evita que una imagen con una
                  # clase muy rara infle demasiado el dataset de entrenamiento

def repeticiones_por_rareza(label_path):
    """Cuántas veces repetir esta imagen en train, según la clase más rara que contenga."""
    if not os.path.exists(label_path):
        return 1
    max_peso = 1.0
    with open(label_path) as f:
        for line in f:
            partes = line.split()
            if len(partes) != 5:
                continue
            class_id = CLASS_ID_MAP.get(int(partes[0]))
            if class_id is None:
                continue  # clase excluida, no influye en el sobremuestreo
            max_peso = max(max_peso, CLASS_WEIGHTS[class_id])
    return min(MAX_REPEATS, max(1, round(max_peso)))

# Split 80/20 sobre las imágenes ORIGINALES, antes del sobremuestreo. Así
# evitamos que copias de la misma imagen terminen repartidas entre train y validation.
RANDOM_SEED = 42
TRAIN_RATIO = 0.8
random.seed(RANDOM_SEED)
image_files_shuffled = list(image_files)
random.shuffle(image_files_shuffled)
n_train_original = int(len(image_files_shuffled) * TRAIN_RATIO)
train_original_paths = image_files_shuffled[:n_train_original]
val_paths = image_files_shuffled[n_train_original:]

# El sobremuestreo solo se aplica a train, nunca a validation.
train_paths = []
for img_path in train_original_paths:
    n_rep = repeticiones_por_rareza(get_label_path(img_path))
    train_paths.extend([img_path] * n_rep)

overlap = set(train_paths) & set(val_paths)
assert len(overlap) == 0, f"DATA LEAKAGE: {len(overlap)} imágenes aparecen en train y validation"

print(f"Imágenes originales: {len(image_files)}")
print(f"Train original: {len(train_original_paths)} | Train tras sobremuestreo: {len(train_paths)}")
print(f"Validation sin sobremuestreo: {len(val_paths)}")


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 1e-4
CHECKPOINT_PATH = "../models/best_detector_6.keras"
RUN_TRAINING = True  # poner en False para dejar todo armado sin arrancar el entrenamiento
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

def load_sample(image_path):
    image_path = image_path.numpy().decode("utf-8")
    label_path = get_label_path(image_path)
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    target = parse_label_to_boxes(label_path)
    return img_array, target

def tf_load_sample(image_path):
    img, target = tf.py_function(load_sample, [image_path], [tf.float32, tf.float32])
    img.set_shape([224, 224, 3])
    target.set_shape([MAX_BOXES, 1 + 4 + N_CLASSES])
    return img, target

train_ds = tf.data.Dataset.from_tensor_slices(train_paths).map(
    tf_load_sample, num_parallel_calls=tf.data.AUTOTUNE
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices(val_paths).map(
    tf_load_sample, num_parallel_calls=tf.data.AUTOTUNE
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Datasets armados")
print("GPUs detectadas:", tf.config.list_physical_devices("GPU"))


In [ ]:
# Fracción de daños reales sobre el total de slots (MAX_BOXES) en un batch.
# Lo usamos para inicializar el bias de objectness cerca de la distribución
# real, en vez de que arranque asumiendo 50/50 entre "hay daño" y "no hay daño".
sample_images, sample_targets = next(iter(train_ds))
frac_positivas = (sample_targets[..., 0] == 1).numpy().mean()
print(f"Fracción de slots con daño real en un batch: {frac_positivas:.4f}")
print(f"O sea, aprox {frac_positivas * MAX_BOXES:.1f} slots positivos de {MAX_BOXES} totales, por imagen")


In [ ]:
# Reutilizamos el modelo 1 (el clasificador daño / no-daño ya entrenado) como
# EXTRACTOR DE FEATURES para este modelo nuevo, en vez de entrenar una red
# desde cero con menos datos. Nos interesa todo lo anterior a
# GlobalAveragePooling: esas capas ya aprendieron a "leer" texturas de daño
# en la imagen. La salida binaria (daño / no-daño) no la usamos acá, la descartamos.
modelo1 = tf.keras.models.load_model("../models/best_model.keras")

backbone_output = None
for layer in modelo1.layers:
    if isinstance(layer, tf.keras.layers.MaxPooling2D):
        backbone_output = layer.output
assert backbone_output is not None, "No se encontró ninguna capa MaxPooling2D en modelo1"

backbone = tf.keras.Model(inputs=modelo1.input, outputs=backbone_output, name="backbone_reutilizado")

# Arranca CONGELADO: en la Fase 1 del entrenamiento (más abajo) solo se
# entrena la cabeza nueva, para no arruinar de entrada estos pesos ya
# entrenados con gradientes grandes de una cabeza que todavía es random.
backbone.trainable = False

print("Backbone reutilizado, forma de salida:", backbone.output_shape)


In [ ]:
from tensorflow.keras import layers

# Cabeza de detección nueva, encima del backbone del modelo 1. A diferencia de
# la versión con grilla, acá NO hay una predicción por celda: el mapa de
# features se colapsa con GlobalAveragePooling2D en un solo vector por imagen,
# y de ahí salen MAX_BOXES "slots" de predicción (objectness, caja, clase),
# cada uno pudiendo representar cualquier daño en cualquier posición.
x = backbone.output  # (14, 14, 256)
x = layers.Conv2D(256, 3, padding="same", activation="relu", name="det_conv2d_1")(x)
x = layers.GlobalAveragePooling2D(name="det_gap")(x)
x = layers.Dense(512, activation="relu", name="det_dense")(x)

# Objectness por slot (MAX_BOXES logits crudos): la loss aplica sigmoid.
bias_inicial_objectness = np.log(frac_positivas / (1 - frac_positivas))
out_obj = layers.Dense(
    MAX_BOXES, activation="linear", name="det_obj",
    bias_initializer=tf.keras.initializers.Constant(bias_inicial_objectness),
)(x)
out_obj = layers.Reshape((MAX_BOXES, 1))(out_obj)

# Caja por slot (xc, yc, w, h): absoluto normalizado 0-1 respecto a la imagen
# completa, acotado con sigmoid. No hay celdas, así que no hay offset relativo.
out_box = layers.Dense(MAX_BOXES * 4, activation="sigmoid", name="det_box")(x)
out_box = layers.Reshape((MAX_BOXES, 4))(out_box)

# Clase por slot: logits, se les aplica softmax en la loss.
out_class = layers.Dense(MAX_BOXES * N_CLASSES, activation="linear", name="det_class")(x)
out_class = layers.Reshape((MAX_BOXES, N_CLASSES))(out_class)

output = layers.Concatenate(axis=-1, name="det_output")([out_obj, out_box, out_class])
modelo_deteccion = tf.keras.Model(inputs=backbone.input, outputs=output, name="modelo_deteccion")
modelo_deteccion.summary()


### Matching Húngaro y loss

Como no hay celdas, no sabemos de antemano qué slot de predicción "le
corresponde" a cada daño real: dos daños distintos pueden salir en cualquier
slot. El matching Húngaro resuelve esto en cada batch: busca la asignación
slot-predicho <-> daño-real que minimiza el error total (caja + clase), y
solo esa asignación se usa para calcular la loss de caja y clase. Es el
mismo enfoque que usa DETR.

In [ ]:
from scipy.optimize import linear_sum_assignment

CLASS_WEIGHTS_TF = tf.constant(CLASS_WEIGHTS, dtype=tf.float32)

def hungarian_match_batch(true_mask, true_boxes, true_classes, pred_boxes, pred_class_probs):
    """Para cada imagen del batch, asigna cada daño real al slot predicho más
    parecido (menor distancia de caja + mayor probabilidad de la clase
    correcta). Devuelve, por slot, el índice del daño real que le tocó
    (-1 si no le tocó ninguno)."""
    batch = true_mask.shape[0]
    match_idx = np.full((batch, true_boxes.shape[1]), -1, dtype=np.int32)
    for b in range(batch):
        valid = true_mask[b] > 0.5
        n_true = int(valid.sum())
        if n_true == 0:
            continue
        tb = true_boxes[b, valid]
        tc = true_classes[b, valid]
        pb = pred_boxes[b]
        pc = pred_class_probs[b]
        cost_box = np.abs(pb[:, None, :] - tb[None, :, :]).sum(axis=-1)
        true_class_idx = tc.argmax(axis=-1)
        cost_class = 1 - pc[:, true_class_idx]
        cost = cost_box + cost_class
        pred_idx, true_idx = linear_sum_assignment(cost)
        real_idx = np.where(valid)[0]
        for p_i, t_i in zip(pred_idx, true_idx):
            match_idx[b, p_i] = real_idx[t_i]
    return match_idx

def detection_loss(y_true, y_pred, lambda_coord=5.0):
    obj_logits = y_pred[..., 0]
    box_pred = y_pred[..., 1:5]
    class_probs = tf.nn.softmax(y_pred[..., 5:], axis=-1)

    match_idx = tf.numpy_function(
        hungarian_match_batch,
        [y_true[..., 0], y_true[..., 1:5], y_true[..., 5:], box_pred, class_probs],
        tf.int32,
    )
    match_idx.set_shape([None, MAX_BOXES])

    # objectness: el slot "debería" predecir 1 si el matching le asignó un
    # daño real, 0 si quedó sin asignar (padding, o sobra de slots)
    obj_target = tf.cast(match_idx >= 0, tf.float32)
    loss_obj = tf.nn.sigmoid_cross_entropy_with_logits(labels=obj_target, logits=obj_logits)

    # caja y clase: SOLO se penalizan en los slots que sí fueron asignados a
    # un daño real (obj_target)
    safe_idx = tf.maximum(match_idx, 0)
    matched_box = tf.gather(y_true[..., 1:5], safe_idx, batch_dims=1)
    matched_class = tf.gather(y_true[..., 5:], safe_idx, batch_dims=1)

    loss_box = tf.reduce_sum(tf.square(matched_box - box_pred), axis=-1)
    loss_class = tf.keras.losses.categorical_crossentropy(matched_class, class_probs)
    peso_clase = tf.reduce_sum(matched_class * CLASS_WEIGHTS_TF, axis=-1)

    loss_box = obj_target * loss_box
    loss_class = obj_target * peso_clase * loss_class

    total = (
        tf.reduce_mean(loss_obj)
        + lambda_coord * tf.reduce_mean(loss_box)
        + tf.reduce_mean(loss_class)
    )
    return total

modelo_deteccion.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=detection_loss,
)


In [ ]:
# Verificación de formas antes de entrenar: el target que entrega el
# dataset tiene que calzar con la salida del modelo.
expected_output_shape = tuple(modelo_deteccion.output_shape[1:])
actual_target_shape = tuple(sample_targets.shape[1:])
print("Batch imágenes:", sample_images.shape)
print("Batch targets:", sample_targets.shape)
print("Salida del modelo:", modelo_deteccion.output_shape)

assert actual_target_shape == expected_output_shape, (
    f"Targets incompatibles con la salida del modelo: {actual_target_shape} != {expected_output_shape}"
)

sample_predictions = modelo_deteccion(sample_images[:1], training=False)
assert tuple(sample_predictions.shape[1:]) == expected_output_shape
print("Verificación OK: train_ds entrega targets compatibles con detection_loss y modelo_deteccion.")


In [ ]:
def crear_callbacks():
    return [
        tf.keras.callbacks.ModelCheckpoint(
            CHECKPOINT_PATH, monitor="val_loss", mode="min", save_best_only=True, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=4, verbose=1,
        ),
    ]

print(f"Callbacks preparados. El mejor detector se guardará en: {CHECKPOINT_PATH}")


### Entrenamiento

Dos fases, una sola pasada de `fit` cada una (en `training3` había dos
celdas de `fit` separadas y redundantes; acá queda un único flujo):
1. **Warm-up** (5 épocas, backbone congelado): solo entrena la cabeza nueva.
2. **Fine-tuning** (hasta `EPOCHS`, backbone descongelado, LR más bajo): ajusta
   toda la red, incluyendo las features heredadas del modelo 1.

Solo corre si `RUN_TRAINING = True` (celda de arriba).

In [ ]:
if not RUN_TRAINING:
    print("Entrenamiento preparado pero no iniciado. Cambiar RUN_TRAINING = True para ejecutar model.fit().")
else:
    print("=== Fase 1: warm-up de la cabeza (backbone congelado) ===")
    history_warmup = modelo_deteccion.fit(
        train_ds,
        validation_data=val_ds,
        epochs=5,
        callbacks=crear_callbacks(),
        verbose=1,
    )

    print("=== Fase 2: fine-tuning completo (backbone descongelado) ===")
    backbone.trainable = True
    modelo_deteccion.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10),
        loss=detection_loss,
    )
    history = modelo_deteccion.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=crear_callbacks(),
        verbose=1,
    )


In [ ]:
if "history" not in globals():
    print("Todavía no hay history: ejecutar primero el entrenamiento.")
else:
    plt.figure(figsize=(8, 5))
    plt.plot(history.history.get("loss", []), label="training loss")
    plt.plot(history.history.get("val_loss", []), label="validation loss")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title("Evolución de la loss del detector (fase 2)")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    best_detector = tf.keras.models.load_model(
        CHECKPOINT_PATH,
        custom_objects={"detection_loss": detection_loss},
    )
    print(f"Mejor detector cargado desde: {CHECKPOINT_PATH}")
else:
    best_detector = modelo_deteccion
    print(f"Aún no existe checkpoint en {CHECKPOINT_PATH}; usando el modelo en memoria.")


### Predicción: clasificar y dibujar el daño en una imagen

En inferencia no hay ningún daño real con el que hacer matching, así que
varios slots pueden disparar para el mismo daño: aplicamos Non-Max
Suppression para quedarnos con una sola caja por daño.

In [ ]:
def calcular_iou(box1, box2):
    """box format: (xc, yc, w, h) normalizado 0-1."""
    x1_min, y1_min = box1[0] - box1[2] / 2, box1[1] - box1[3] / 2
    x1_max, y1_max = box1[0] + box1[2] / 2, box1[1] + box1[3] / 2
    x2_min, y2_min = box2[0] - box2[2] / 2, box2[1] - box2[3] / 2
    x2_max, y2_max = box2[0] + box2[2] / 2, box2[1] + box2[3] / 2

    inter_xmin = max(x1_min, x2_min)
    inter_ymin = max(y1_min, y2_min)
    inter_xmax = min(x1_max, x2_max)
    inter_ymax = min(y1_max, y2_max)

    inter_area = max(0, inter_xmax - inter_xmin) * max(0, inter_ymax - inter_ymin)
    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]
    union_area = area1 + area2 - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def non_max_suppression(predicciones, umbral_iou_nms=0.4):
    """predicciones: lista de (xc, yc, w, h, class_id, objectness)."""
    resultado = []
    clases_presentes = set(p[4] for p in predicciones)
    for clase in clases_presentes:
        preds_clase = [p for p in predicciones if p[4] == clase]
        preds_clase = sorted(preds_clase, key=lambda p: p[5], reverse=True)
        while preds_clase:
            mejor = preds_clase.pop(0)
            resultado.append(mejor)
            preds_clase = [p for p in preds_clase if calcular_iou(mejor[:4], p[:4]) < umbral_iou_nms]
    return resultado


def predecir_cajas(modelo, image_path, umbral_objectness=0.3, umbral_iou_nms=0.4, img_size=IMG_SIZE):
    """Corre el modelo sobre una imagen y devuelve las cajas predichas
    (xc, yc, w, h, class_id, objectness), ya filtradas por NMS."""
    img = Image.open(image_path).convert("RGB").resize(img_size)
    img_array = tf.keras.utils.img_to_array(img)
    pred = modelo(tf.expand_dims(img_array, 0), training=False)[0]  # (MAX_BOXES, 1+4+N_CLASSES)

    predicciones = []
    for slot in range(pred.shape[0]):
        objectness = tf.sigmoid(pred[slot, 0]).numpy()
        if objectness < umbral_objectness:
            continue
        xc, yc, w, h = pred[slot, 1:5].numpy()
        class_id = int(tf.nn.softmax(pred[slot, 5:]).numpy().argmax())
        predicciones.append((xc, yc, w, h, class_id, objectness))

    return non_max_suppression(predicciones, umbral_iou_nms)


def plot_pred_box_on_image(label_path, ax, umbral_objectness=0.3):
    """Dibuja las cajas PREDICHAS por el modelo: mismo estilo e interfaz
    (recibe label_path) que plot_real_box_on_image, para comparar fácil."""
    img_name = os.path.basename(label_path).replace(".txt", "")
    img_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = f"{LOCAL_DATA_DIR}/images/{img_name}{ext}"
        if os.path.exists(candidate):
            img_path = candidate
            break

    cajas = predecir_cajas(best_detector, img_path, umbral_objectness=umbral_objectness)
    img = Image.open(img_path)
    img_w, img_h = img.size
    ax.imshow(img)

    for xc, yc, w, h, class_id, objectness in cajas:
        box_w = w * img_w
        box_h = h * img_h
        x0 = xc * img_w - box_w / 2
        y0 = yc * img_h - box_h / 2

        rect = patches.Rectangle((x0, y0), box_w, box_h,
                                  linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x0, y0 - 4, f"{CLASS_NAMES[class_id]} ({objectness:.2f})",
                color="red", fontsize=7, weight="bold")

    ax.set_title(f"Predicción: {img_name}", fontsize=9)
    ax.axis("off")


### Demo: label random con daño, real vs. predicho

Elegimos un label al azar (de validation, así el modelo no lo vio en
entrenamiento) que tenga al menos un daño real, y comparamos la anotación
original contra lo que predice el modelo.

In [ ]:
labels_val_con_dano = [
    get_label_path(p) for p in val_paths
    if parse_label_to_boxes(get_label_path(p))[:, 0].sum() > 0
]
label_demo = random.choice(labels_val_con_dano)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_real_box_on_image(label_demo, axes[0])
axes[0].set_title(f"Real: {os.path.basename(label_demo)}", fontsize=10)
plot_pred_box_on_image(label_demo, axes[1], umbral_objectness=0.3)
plt.tight_layout()
plt.show()


### (Opcional) Evaluación cuantitativa

Precision / recall / IoU promedio sobre todo el set de validation, a
distintos umbrales de objectness.

In [ ]:
def evaluar_deteccion(modelo, paths, umbral_objectness=0.5, umbral_iou=0.5):
    total_predicciones = total_reales = verdaderos_positivos = 0
    ious_matched = []

    for image_path in paths:
        predicciones = predecir_cajas(modelo, image_path, umbral_objectness=umbral_objectness)

        boxes_reales = parse_label_to_boxes(get_label_path(image_path))
        valid = boxes_reales[:, 0] == 1
        reales = [
            (xc, yc, w, h, int(np.argmax(class_onehot)))
            for xc, yc, w, h, *class_onehot in boxes_reales[valid, 1:]
        ]

        total_predicciones += len(predicciones)
        total_reales += len(reales)

        usados = set()
        for pred_box in predicciones:
            mejor_iou, mejor_idx = 0, -1
            for idx, real_box in enumerate(reales):
                if idx in usados:
                    continue
                iou = calcular_iou(pred_box[:4], real_box[:4])
                if iou > mejor_iou:
                    mejor_iou, mejor_idx = iou, idx
            if mejor_iou >= umbral_iou:
                verdaderos_positivos += 1
                ious_matched.append(mejor_iou)
                usados.add(mejor_idx)

    precision = verdaderos_positivos / total_predicciones if total_predicciones else 0.0
    recall = verdaderos_positivos / total_reales if total_reales else 0.0
    iou_promedio = np.mean(ious_matched) if ious_matched else 0.0

    print(f"Umbral objectness={umbral_objectness} | Predicciones: {total_predicciones} | "
          f"Reales: {total_reales} | TP (IoU>={umbral_iou}): {verdaderos_positivos}")
    print(f"  Precision: {precision:.3f} | Recall: {recall:.3f} | IoU promedio: {iou_promedio:.3f}")
    return precision, recall, iou_promedio


for umbral in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    evaluar_deteccion(best_detector, val_paths, umbral_objectness=umbral)
